In [28]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import (
    FunctionTransformer,
    StandardScaler,
    OneHotEncoder
)


np.random.seed(1234)


# --------------------------------------------------
# Custom transformers (unchanged)
# --------------------------------------------------

class SelectiveStandardScaler(BaseEstimator, TransformerMixin):
    def __init__(self, cols):
        self.cols = cols
        self.scaler = StandardScaler()

    def fit(self, X, y=None):
        self.scaler.fit(X[self.cols])
        return self

    def transform(self, X):
        X = X.copy()
        X[self.cols] = self.scaler.transform(X[self.cols])
        return X


def aggregate_by_observation(df):
    cat_cols = ["cat_0", "cat_1", "cat_2", "cat_3", "cat_4"]

    inconsistent = (
        df.groupby("obs")[cat_cols]
          .nunique()
          .ne(1)
          .any()
    )
    if inconsistent.any():
        bad_cols = inconsistent[inconsistent].index.tolist()
        raise ValueError(f"Inconsistent categorical values per obs: {bad_cols}")

    numeric_cols = df.select_dtypes(include="number").columns

    agg = (
        df.groupby("obs", sort=False)[numeric_cols]
          .mean()
          .drop(columns="obs")
    )
    return agg


# --------------------------------------------------
# Column definitions
# --------------------------------------------------

NUMERIC_SCALE_COLS = [
    "num_0", "num_1", "num_2",
    "t_0", "t_1", "t_2", "t_3", "t_4"
]

CAT_ALL = ["cat_0", "cat_1", "cat_2", "cat_3", "cat_4"]
CAT_ONEHOT = ["cat_1"]
CAT_BINARY = ["cat_0", "cat_2", "cat_3", "cat_4"]


# --------------------------------------------------
# Base preprocessing (time aggregation etc.)
# --------------------------------------------------

base_pipeline = Pipeline([
    (
        "coerce_numeric",
        FunctionTransformer(
            lambda df: df.apply(pd.to_numeric, errors="coerce"),
            validate=False
        )
    ),
    (
        "aggregate_obs",
        FunctionTransformer(aggregate_by_observation, validate=False)
    ),
    (
        "scale_selected",
        SelectiveStandardScaler(NUMERIC_SCALE_COLS)
    )
])


# --------------------------------------------------
# Column transformer (NEW)
# --------------------------------------------------

feature_encoder = ColumnTransformer(
    transformers=[
        ("cat1_ohe", OneHotEncoder(sparse_output=False, handle_unknown="ignore"), CAT_ONEHOT),
        ("cat_binary", "passthrough", CAT_BINARY),
        ("numeric", "passthrough", [c for c in NUMERIC_SCALE_COLS]),
    ],
    remainder="drop"
)


# --------------------------------------------------
# Full feature pipeline
# --------------------------------------------------

X_pipeline = Pipeline([
    ("base", base_pipeline),
    ("encode", feature_encoder)
])


# --------------------------------------------------
# Load + preprocess
# --------------------------------------------------

df = pd.read_csv("Project_Data_export/train_competition_2026.csv")
df.sort_values("time", inplace=True)

time_by_obs = df.groupby("obs")["time"].first()

processed = base_pipeline.fit_transform(df)

processed["time"] = processed.index.map(time_by_obs)
processed.sort_values("time", inplace=True)

assert processed["time"].is_monotonic_increasing


# --------------------------------------------------
# Split
# --------------------------------------------------

X = processed.drop(columns=["sub_id", "time", "y_1", "y_2"])
y = processed[["y_1", "y_2"]]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=1234,
    shuffle=False
)


# --------------------------------------------------
# Apply feature pipeline
# --------------------------------------------------

X_train = feature_encoder.fit_transform(X_train)
X_test  = feature_encoder.transform(X_test)

print("Preprocessing complete.")

Preprocessing complete.
